# 05 — Inference backend for the web front end (ONLINE, **GPU P100**)

| attach as input | produces |
|---|---|
| `behaviorsense-code`, `behaviorsense-runs` | a public HTTPS URL you paste into the front end |
| Kaggle Model: **Qwen2.5-7B-Instruct** | only if Agent 4 runs locally - see below |

## Pick P100, not T4 x2, and not a TPU

**P100.** With Agent 4 hosted, nothing left on this GPU uses tensor cores: `rtmo-l.onnx` is an
fp32 export, and so are OSNet and ST-GCN++. On fp32 the cards invert - T4 is 8.1 TFLOPS at
320 GB/s, P100 is 9.3 TFLOPS at **732 GB/s** - and pose extraction is bandwidth-bound. T4's 65
TFLOPS of fp16 tensor throughput only ever mattered for Qwen. One card also removes the
pipeline-parallel PCIe hop that had both T4s reading 0% utilisation while the CPU spun.

VRAM stops being the constraint: RTMO's session is ~1-2 GB, OSNet ~0.5 GB, ST-GCN++ ~50 MB.
Under 4 GB of 16. The two T4s were only ever needed because a 7B model would not fit on one.

**Not a TPU**, whatever its RAM. `onnxruntime` has no TPU execution provider, so RTMO - the one
genuinely GPU-bound stage - cannot run on it at all and would fall back to CPU. TPUs are also
throughput devices, and this is batch-of-one interactive inference: the shape they are worst at.

## Agent 4 runs off-box by default

Set **`OPENROUTER_API_KEY_LIST`** in Add-ons -> Secrets (comma-separated; numbered
`OPENROUTER_API_KEY_1..N` also work) and the local 7B is never loaded - no 70 s load, no 17.8 GiB,
no 286 s report. Without keys it falls back to Qwen and says so.

Qwen stays the MEASURED arm: every figure in `results/evaluation.md` came from it under
grammar-constrained decoding on a pinned checkpoint. A `:free` endpoint can be deprecated without
notice, so it is the demo arm, and `/health` reports `agent4` so the page states which model
wrote a report.

**Internet ON, GPU on.** This notebook is the only one that serves rather than computes: it
loads the trained ADL streams and Qwen behind FastAPI, opens a Cloudflare quick tunnel, and
prints the address. `web/` then talks to that address from anywhere.

Why a tunnel rather than hosting the model near the front end: Qwen2.5-7B needs ~16 GB in
bf16, and the ADL ensemble needs a GPU to be worth calling at all. Neither fits a laptop, and
quantising to fit would trade the exact numbers `results/evaluation.md` reports for ones
nobody has measured. Kaggle's T4 x2 / P100 are free and already hold every weight.

**The address changes every session.** That is inherent to a quick tunnel with no account,
and it is why the front end asks you to paste it rather than hard-coding one. Run this
notebook, copy the line it prints, paste it into the field in the header.

**This API is unauthenticated.** Anyone with the URL can post to it for as long as the
session lives. That is acceptable for a demo you start and stop deliberately; it is not a
deployment. `BS_TOKEN` below turns on a shared-secret header if you want one — the front end
has a field for it.

Deployment configuration is not a guess: `logit_adjust_tau=0.25` and the fitted transition
prior are the settings the notebook-04 lever table selected, and `do_sample=False` is the
greedy decoding the reporter requires. Serving anything else would mean the demo and the
measurements describe different systems.

In [ ]:
# Resolve the repo mount by CONTENT, not by dataset name.
#
# The dataset title is free text and this project has already been uploaded under more
# than one spelling ("behaviorsense-*" and "behavioursense-*"). Hard-coding the name makes
# cell 1 of a 12-hour session fail on a typo, so find the repo by a file only it contains.
import pathlib
from itertools import islice

INPUT = pathlib.Path("/kaggle/input")
def attached_mounts():
    # Datasets do NOT sit directly under /kaggle/input. They mount at
    # /kaggle/input/datasets/<owner>/<name>/, and competitions at
    # /kaggle/input/competitions/<name>/. Listing INPUT.iterdir() therefore always
    # reports ['competitions', 'datasets'] whatever is attached - which is what the
    # "code: nothing matches ..." failure printed, telling us nothing about whether
    # the code dataset was attached. Descend to the level that names real mounts, and
    # report what each one CONTAINS, since a dataset can be attached and still be
    # missing the directory the notebook needs.
    out = []
    for container in ("datasets", "competitions"):
        base = INPUT / container
        if not base.is_dir():
            continue
        for owner in sorted(base.iterdir()):
            kids = sorted(owner.iterdir()) if owner.is_dir() else []
            if kids and all(k.is_dir() for k in kids[:1]) and container == "datasets":
                for ds in kids:
                    # islice, NOT sorted(...)[:6]. `sorted()` materialises the whole listing
                    # first, and on Kaggle's FUSE mount a slug whose files sit at its root -
                    # `toyota-smarthome-skeleton-v1-2` holds 16,115 - makes that a full
                    # network directory read per mount. Eleven mounts of that shape is most
                    # of the 14 minutes this cell took on the first Toyota run. Six names
                    # are all this diagnostic needs, so stop after six.
                    top = sorted(islice((q.name for q in ds.iterdir()), 6)) \
                        if ds.is_dir() else []
                    out.append(f"{ds.name} (top level: {top})")
            else:
                out.append(owner.name)
    # Fall back to the flat layout so this keeps working if Kaggle changes the mount
    # shape back, rather than reporting nothing at all.
    return out or sorted(p.name for p in INPUT.iterdir())

ATTACHED = attached_mounts() if INPUT.is_dir() else []

# MOUNT-INDEXED SEARCH, and why two earlier fixes were not enough.
#
# `INPUT.glob("**/x")` walks every directory under /kaggle/input - 93 minutes with the
# Toyota corpus mounted, notebook 06 measured, and 349 s for the single
# `**/src/behaviorsense/__init__.py` probe in notebook 07's second run. The first "fix",
# FIXED-DEPTH globs like `datasets/*/*/*/rtmo-l.onnx`, was depth-bounded but not
# COST-bounded: to match at depth 3 pathlib scandirs EVERY slug child directory, including
# `toyota-smarthome-skeleton-v1-2`'s 16,115-file root and MSMT17's 65,242 crops.
#
# The mounts are KNOWN at depth 2 (`datasets/<owner>/<slug>/`), so enumerate them once and
# resolve everything else with is_file() stats - one metadata call per mount per candidate,
# never a sibling-directory listing.
_SLUGS = (sorted((INPUT / "datasets").glob("*/*"))
          + sorted((INPUT / "competitions").glob("*")))

def find_fast(tail, what, required=True):
    # Known staging prefixes, each costing one stat per mount:
    #   ""                      files at the slug root
    #   EmotionSense-Extended/  the code dataset was created by zipping the repo FOLDER,
    #                           so everything sits one level below the slug
    #   kaggle/working/         Save Version nests the working directory
    # The prefixes apply to MULTI-COMPONENT tails too. They used to be tried only for bare
    # filenames, which quietly sent `src/behaviorsense/__init__.py` - the one probe every
    # notebook makes - down the deep-search path it was written to avoid.
    # `weights/` is additionally tried for a bare filename, the staged weights layout.
    prefixes = ("", "EmotionSense-Extended/", "kaggle/working/")
    mids = ("",) if "/" in tail else ("", "weights/")
    for t in [p + m + tail for p in prefixes for m in mids]:
        hits = [s / t for s in _SLUGS if (s / t).is_file()]
        if hits:
            return hits[0]
    hits = sorted(INPUT.glob(f"**/{tail}"))   # last resort: unusual layout, slow, once
    if hits:
        print(f"  {what:<9} found only by deep search ({tail}) - layout is unusual")
        return hits[0]
    if required:
        raise AssertionError(
            f"{what}: nothing matches {tail!r} in any mount. Attached: {ATTACHED}")
    print(f"  {what:<9} ABSENT (optional)")
    return None

def find_asset(pattern, what, required=True):
    # Callers pass a `**/...` pattern. The leading `**/` is stripped and the mount-indexed
    # search runs first, so every existing call site gets the speed-up unchanged.
    tail = pattern[3:] if pattern.startswith("**/") else pattern
    return find_fast(tail, what, required=required)

def find_dir(subdir, pattern, roots=None):
    # "Which mount holds the most files matching this pattern in this subdirectory?" - one
    # scandir of ONE named directory per mount, never a recursive walk. Used for corpora
    # (Toyota's mp4/ and Videos_mp4/) where the answer is a directory, not a file.
    from fnmatch import fnmatch
    import os
    best, best_n = None, 0
    for root in (roots if roots is not None else _SLUGS):
        base = root / subdir if subdir else root
        if not base.is_dir():
            continue
        n = sum(1 for e in os.scandir(base) if e.is_file() and fnmatch(e.name, pattern))
        if n > best_n:
            best, best_n = base, n
    return best, best_n

SRC     = find_asset("**/src/behaviorsense/__init__.py", "code").parent.parent
CODE    = SRC.parent
SCRIPTS = CODE / "scripts"
CONFIGS = CODE / "configs"
import sys
sys.path.insert(0, str(SRC)); sys.path.insert(0, str(SCRIPTS))
print(f"  code {CODE}")
print(f"  attached {ATTACHED}")

CONTRACT = [
    ("scripts/kaggle_smoke_test.py", "--profile",      "notebook 03 preflight"),
    ("scripts/train_adl.py",         "--stop-after",   "resume guard (notebook 03)"),
    ("scripts/train_adl.py", "--tau-train",
     "notebook 03 passes --sampler/--tau-train; a snapshot predating them exits 2 from "
     "argparse, which notebook 03 reports as a failed stream rather than stale code"),
    ("src/behaviorsense/data/skeleton_dataset.py", "def load_subject_map",
     "video-id -> Charades actor-id remap for a person-disjoint P1 split (notebooks "
     "03/04). A stale snapshot silently reverts P1 to video-disjoint - same person in "
     "train and val - while printing numbers that look identical"),
    ("scripts/train_fall.py", "pos_rate > 0.5 and args.focal_alpha > 0.5",
     "refuses focal alpha that up-weights the majority; a snapshot without it trains "
     "the fall head on 82% positives and reports a plausible but meaningless AUPRC"),
    ("scripts/prepare_skeletons.py", "def assign_slots",
     "slot tracking + windowing (notebooks 01/02)"),
    ("scripts/prepare_skeletons.py", "with_starts",
     "fall labelling by true frame position (notebook 02); index-derived position "
     "mislabels the descent whenever a window is dropped"),
    ("scripts/prepare_skeletons.py", "def le2i_fall_frames",
     "Le2i has no fall/ADL marker in any path component; without this its ~192 fall "
     "clips land in the negatives (notebook 02)"),
    ("scripts/prepare_skeletons.py", "def label_fall_windows",
     "shared fall labelling: exact interval for Le2i, positional fallback elsewhere"),
    ("src/behaviorsense/data/skeleton_dataset.py", "keep_root_motion",
     "falls are unlearnable without it"),
    ("src/behaviorsense/models/ensemble.py", "def per_stream_logits", "notebook 04 ablation"),
    ("src/behaviorsense/models/ensemble.py", "def flip_windows",
     "test-time flip augmentation (notebook 04 levers cell). A stale snapshot would accept "
     "`clf.tta = True` as a new attribute and silently do no TTA, reporting the "
     "unaugmented number as if it were augmented"),
    ("src/behaviorsense/models/stgcnpp.py", "parents.setdefault",
     "flip-equivariant bone stream; without it half the ensemble trains on sign noise. The "
     "token tracked the local name `parent` and broke when the function grew an explicit "
     "`parents` argument for Toyota's 15-node tree - a rename silently disarming a staleness "
     "guard is exactly what this list exists to catch, so it caught itself"),
    ("src/behaviorsense/kaggle_artifacts.py", "def find_run_dir",
     "one rule for 'is this real session output or a dev leftover'; four call sites "
     "learned it separately and the fourth was missed"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def force_greedy",
     "pins BOTH decoding arms to greedy (notebook 04). Without it the constrained arm "
     "inherits Qwen's generation_config (do_sample=True, temperature=0.7) while the free "
     "arm is greedy, so the comparison measures temperature instead of grammar - the "
     "constrained rate moved 8.2% -> 10.3% between two runs of identical code"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def _chat_text",
     "both arms send the SAME templated text (notebook 04). The constrained path used to "
     "hand outlines the raw prompt, so one arm got a Qwen chat turn and the other a naked "
     "instruction block - a second confound on top of the sampling one"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def repair_claim",
     "format-only claim repair + maxItems bound to max_claims (notebook 04). Without "
     "it the unconstrained arm scores 245 emitted / 0 scorable / nan%, which measures "
     "JSON compliance rather than faithfulness"),
    ("scripts/eval_hallucination.py", "unusable_rate",
     "three-arm hallucination table with a denominator over EMITTED claims; a stale "
     "snapshot silently reports the two-arm nan% version"),
    ("src/behaviorsense/eval/activity_eval.py", "def logit_adjust",
     "notebook 04's P1 cell imports MIN_SUPPORT and scores() from here, so a stale "
     "snapshot fails with ImportError at cell 3; also carries the post-hoc accuracy "
     "levers scripts/rescore_p1.py replays off the saved val logits"),
    ("src/behaviorsense/video.py", "def child_env",
     "notebook 05's /video endpoint decodes uploads in a CHILD process (ffmpeg raises SIGSEGV "
     "on malformed streams and a signal is not catchable, so without the boundary one bad "
     "upload kills the kernel, the tunnel and the demo together). `child_env` is what puts "
     "behaviorsense on that child's PYTHONPATH - sys.path does not cross a process boundary, "
     "and a snapshot without it 422s EVERY upload with \"No module named 'behaviorsense'\""),
    ("src/behaviorsense/pipeline.py", "def frames_to_windows", "Agent 1 -> Agent 2 seam"),
    ("configs/taxonomy.yaml",        None,             "class map (notebook 01)"),
]
_stale = []
for _rel, _token, _why in CONTRACT:
    _p = CODE / _rel
    if not _p.is_file():
        _stale.append(f"{_rel} is MISSING ({_why})")
        continue
    if _token and _token not in _p.read_text(encoding="utf-8", errors="ignore"):
        _stale.append(f"{_rel} lacks {_token!r} ({_why})")
if _stale:
    raise AssertionError(
        "The attached code dataset is OLDER than these notebooks:\n  - "
        + "\n  - ".join(_stale)
        + f"\n\nMounted: {CODE}\nRe-upload from your checkout, then restart this notebook:"
          "\n  kaggle datasets version -p . --dir-mode zip -m \"sync\""
    )
print(f"  contract {len(CONTRACT)}/{len(CONTRACT)} - mounted code is current")

In [ ]:
# Deps. `cloudflared` is a single static binary - no account, no config file. Pinned to a
# release rather than `latest` so a breaking change upstream cannot silently take the demo
# down: the URL format this notebook greps for is part of that contract.
#
# onnxruntime-gpu is PINNED to 1.26.0 and the pin is load-bearing. From 1.27 the PyPI GPU
# wheels are built against CUDA 13 while Kaggle's image is CUDA 12; the provider is still
# LISTED by get_available_providers() and then fails to load, so RTMO runs on CPU at roughly
# 1/50th speed and the only symptom is a video that takes forever. video.py asserts on the
# session's providers, which is the only honest source.
import subprocess, sys, os, pathlib

# ORDER IS LOAD-BEARING, and this cell had it backwards. `rtmlib` depends on the CPU
# `onnxruntime`, and the CPU and GPU packages own the SAME `onnxruntime/` directory. The
# previous sequence installed everything - pip pulling the CPU build in as rtmlib's
# dependency - and then uninstalled `onnxruntime` afterwards, which deletes files SHARED
# with onnxruntime-gpu. The result imports and is hollow:
#
#   AttributeError: module 'onnxruntime' has no attribute '__version__'
#
# The comment that used to be here claimed removing the CPU build after the install avoided
# the GPU one being "shadowed". That was wrong, and notebooks 01 and 07 already do it the
# right way round - three successful extraction runs prove the sequence below:
#
#   1. remove any CPU onnxruntime FIRST
#   2. install rtmlib with --no-deps so it cannot drag the CPU build back in
#   3. install onnxruntime-gpu LAST, so nothing writes over its provider registration
#
# onnxruntime-gpu is PINNED to 1.26.0 and the pin is load-bearing. From 1.27 the PyPI GPU
# wheels are built against CUDA 13 while Kaggle's image is CUDA 12; the provider is still
# LISTED by get_available_providers() and then fails to load, so RTMO runs on CPU at roughly
# 1/50th speed and the only symptom is a video that takes forever. video.py asserts on the
# session's providers, which is the only honest source.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime"],
               check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "rtmlib"],
               check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "fastapi", "uvicorn", "nest_asyncio", "python-multipart",
                "outlines>=1.0", "opencv-python-headless"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "onnxruntime-gpu==1.26.0"], check=True)

CF = pathlib.Path("/usr/local/bin/cloudflared")
if not CF.exists():
    url = ("https://github.com/cloudflare/cloudflared/releases/download/2024.12.2/"
           "cloudflared-linux-amd64")
    subprocess.run(["curl", "-fsSL", "-o", str(CF), url], check=True)
    CF.chmod(0o755)
print(subprocess.run([str(CF), "--version"], capture_output=True, text=True).stdout.strip())

# Check the install is WHOLE, not just importable. A missing `__version__` is the signature
# of the shared-directory breakage above, so it gets a message that names the cause instead
# of an AttributeError six lines into a demo.
import onnxruntime
_ver = getattr(onnxruntime, "__version__", None)
assert _ver is not None, (
    "onnxruntime imported but has no __version__ - the package directory is incomplete, "
    "which happens when the CPU build is uninstalled AFTER onnxruntime-gpu (they share "
    "the directory). Restart the kernel and re-run this cell: it now removes the CPU build "
    "first and installs the GPU build last.")
_prov = onnxruntime.get_available_providers()
print("onnxruntime", _ver, _prov)
assert any("CUDA" in p for p in _prov), (
    f"no CUDAExecutionProvider in {_prov}. RTMO would run on CPU at ~1/50th speed and the "
    "only symptom would be a very slow upload.")

In [ ]:
# Resolve the weights. Same content-based rule as every other notebook: never trust a
# dataset name. find_run_dir picks the directory holding adl_<stream>/ and skips leftovers
# inside a mounted code checkout - the trap that cost notebook 04 a session.
import glob, os
from behaviorsense.kaggle_artifacts import find_run_dir

RUNS = str(find_run_dir(INPUT))
# Per-mount, not a `**` walk. `**/config.json` visits every file under /kaggle/input -
# including MSMT17's 65,242 crops inside behaviorsense-code - to find one file in a
# directory whose name already says "qwen".
_qwen = [c for s in _SLUGS if "qwen" in s.name.lower()
         for c in ((s / "config.json"), *(s.glob("*/config.json"))) if c.is_file()]
assert _qwen, f"attach the Qwen2.5-7B-Instruct Kaggle Model. Attached: {ATTACHED}"
QWEN = str(_qwen[0].parent)

# Pose + re-id weights for the video path. RTMO is required for it; OSNet is optional and
# its absence degrades roles to UNKNOWN rather than to a guess.
_rtmo_p = find_asset("**/rtmo-l.onnx", "rtmo", required=False)
RTMO = str(_rtmo_p) if _rtmo_p else None
_osnet_p = find_asset("**/osnet_ain_x1_0_msmt17.pth", "osnet", required=False)
# OSNet weights. Enablement is PER REQUEST (see `use_osnet` in /video), because matching
# against an enrolled gallery. So every track returns `unidentified` whether it runs or not - the
# `subject of Agent 3's features` badge comes from the most-present-track assertion, not from
# re-identification. On CPU (which a P100 forces, since torch has no sm_60 kernels) it pushed
# 1,202 person crops through a ReID CNN and took pose extraction from 29.8 s to 134.2 s to produce
# the string "unknown". Measured, so it stays off until something actually enrols someone.
# PER-REQUEST, not per-notebook. OSNet used to be off unconditionally: with no enrolled
# resident it could only return `unidentified`, and on CPU (which a P100 forces - torch has
# no sm_60 kernels) it pushed 1,202 crops through a ReID CNN for 104 s to produce that
# string. The gallery changes the economics: when a request CARRIES one (the local backend
# attaches the operator's gallery.json, see web/local_backend.py) or asks to enrol, matching
# can actually answer; sampling (reid_embed_interval) brings the CPU cost from 104 s to a
# few seconds. No gallery and no enrol request -> OSNet stays off, exactly as before.
OSNET = str(_osnet_p) if _osnet_p else None
if _osnet_p:
    print(f"  osnet found at {OSNET}: enabled per request when a gallery or an enrol "
          "name is present (embedding sampled every "
          "reid_embed_interval frames, not every frame)")
else:
    print("  osnet ABSENT: re-identification stays off and identity is asserted, "
          "not recognised")

# PREFLIGHT THE DECODE CHILD, here, rather than discovering it on the first upload.
#
# Decode runs in a separate interpreter, and `sys.path` does not cross a process boundary.
# The resolver cell above made `behaviorsense` importable in THIS process; a fresh one knows
# nothing about it. That is a real failure this notebook shipped: an uploaded clip came back
# `pose extraction failed: No module named 'behaviorsense'` while the parent was importing
# the package perfectly well. `behaviorsense.video.child_env` now puts the package on the
# child's PYTHONPATH, derived from the module's own location.
#
# It is preflighted because of WHEN it would otherwise fail. Every other prerequisite here is
# checked at startup; this one used to surface mid-demo, on someone else's video, as a 422
# that read like a bad file. A one-second probe now converts that into a failure at the point
# where it can still be fixed.
if RTMO:
    import subprocess as _sp
    from behaviorsense.video import PKG_PARENT as _PKGP, child_env as _cenv
    _probe = _sp.run([sys.executable, "-c", "import behaviorsense.video as v; print(v.__file__)"],
                     capture_output=True, text=True, env=_cenv(), timeout=120)
    assert _probe.returncode == 0, (
        "the decode CHILD cannot import behaviorsense, so every /video upload would 422.\n"
        f"  PYTHONPATH given to the child: {_cenv()['PYTHONPATH']}\n"
        f"  package expected under:        {_PKGP}\n"
        f"  child stderr: {_probe.stderr.strip()[-400:]}\n"
        "  FIX: re-upload `behaviorsense-code` from your checkout - this needs the version of "
        "src/behaviorsense/video.py that sets the child's PYTHONPATH (it defines `child_env`).")
    print(f"  decode child OK -> {_probe.stdout.strip()}")
else:
    print("  rtmo-l.onnx not attached: /video is off, decode child not probed")

# Serve the configuration notebook 04 SELECTED, not the defaults. Every value here is a
# measured choice from results/evaluation.md, and each one was wrong at some point:
#
#   STREAMS   the 4-stream ensemble was the headline and is the WORST credible option on
#             macro-F1 (0.128 vs 0.146 for `bone` alone). macro-F1 is the arbiter because
#             Agent 3 turns window predictions into daily durations, where over-predicting
#             a class inflates a duration exactly as missing one deflates it.
#   TEMPERATURE  must be fitted FOR THE SERVED STREAMS. The 0.63 in the run log was fitted
#             on the 4-stream ensemble's logits; serving `bone` alone with it would be
#             miscalibrated, and Viterbi plus abstention both consume those posteriors.
#             So it is refitted here from the saved val logits - free, on CPU.
#   TAU       0.25, swept in notebook 04 with tau=0 asserted as an exact no-op.
STREAMS = ("bone",)                 # best macro-F1; ("bone", "joint") for best mean-class
TAU = 0.25
# SAMPLE_FPS belongs in this block for the same reason STREAMS and TAU do: it is a property of
# the CHECKPOINT, not of the camera, and getting it wrong is silent.
#
# A 30-frame window is `30 / SAMPLE_FPS` seconds of motion, and ST-GCN++ has only ever seen the
# duration its training shards were built at. The corpora do not agree:
#
#   Charades (notebook 01)  FPS_SAMPLE = 15  ->  2.00 s   <- what runs/adl_bone/best.pt saw
#   fall     (notebook 02)  fps_sample = 15  ->  2.00 s
#   Toyota   (notebook 07)  FPS_SAMPLE = 20  ->  1.50 s
#
# So this must be flipped to 20.0 AT THE SAME TIME as the Toyota-trained checkpoint is served,
# not before and not after. Serving 20 Hz against the Charades checkpoint would stretch every
# action by 1.33x - the same defect as the old `src_fps / 2` stride, just moved to a constant.
# The rate travels in the /video payload and the page draws a banner when it is not 15, so a
# mismatch is visible rather than absorbed as bad accuracy.
# The rate to use when the checkpoint does not record one. It MUST agree with ADL_PREFER
# below: they are two constants in this cell describing one choice, and they were
# contradicting each other - ADL_PREFER said "toyota" while this said None, so the loader
# fail-closed on every start and the notebook could not serve the checkpoint it was
# configured for.
#
# `None` remains available and means DERIVE-OR-REFUSE. It is the right value once every
# checkpoint records its own rate (`train_adl.py` now does), and the wrong one while a
# pre-fix checkpoint is still being served - refusing to start is only useful if there is
# something the operator can do about it, and here the answer was already known.
#
#   20.0  Toyota RTMO shards (1.50 s windows) - matches ADL_PREFER = "toyota"
#   15.0  Charades shards    (2.00 s windows) - set this if you serve `adl_bone/`
SAMPLE_FPS = 20.0
# Fallback only. The loader below DERIVES the rate from the served checkpoint's own shard paths
# (`..._20hz_...`) and writes it to STATE["sample_fps"]; this value is used only when the shard
# names carry no rate at all.
#
# ADL_PREFER disambiguates when several runs are attached for the same stream - `adl_bone/`
# (Charades, 20 classes, 15 Hz) and `adl_toyota_bone/` (Toyota RTMO, 22 classes, 20 Hz) both
# match stream `bone`. Set to "toyota" to serve the corpus filmed in a real home on mounted
# cameras; set to "" to take whatever sorts first and accept the coin toss.
ADL_PREFER = "toyota"
MAX_UPLOAD_MB = 60
TOKEN = os.environ.get("BS_TOKEN", "")      # optional shared secret; "" disables the check

# Refit T for the streams actually being served. val_logits.npz is written by notebook 04;
# without it a hardcoded temperature silently belongs to a different model.
TEMPERATURE = None
_vl_p = find_asset("**/val_logits.npz", "val_logits", required=False)
_vl = [_vl_p] if _vl_p else []
if _vl:
    import numpy as _np
    from behaviorsense.agents.activity import fit_temperature as _fitT
    from behaviorsense.eval.activity_eval import combine as _combine
    with _np.load(_vl[0], allow_pickle=False) as _z:
        _per = {k[len("logits_"):]: _z[k] for k in _z.files
                if k.startswith("logits_") and k != "logits_ensemble"}
        _y = _z["y"]
    _missing = set(STREAMS) - set(_per)
    assert not _missing, f"val_logits.npz lacks {sorted(_missing)}; has {sorted(_per)}"
    TEMPERATURE = float(_fitT(_combine(_per, tuple(STREAMS), "logit"), _y))
    # The head size these logits describe. `val_logits.npz` carries no link to the checkpoint
    # that produced it, and the first served run fitted T=0.79 from CHARADES logits (20 classes,
    # 35,698 windows) onto the 22-class Toyota model without a murmur - the exact failure the
    # comment above warns about. `_load` compares this against the served head and discards the
    # fit if they disagree, because a temperature from another model is worse than none.
    TEMPERATURE_N_CLASSES = int(next(iter(_per.values())).shape[1])
    TEMPERATURE_SOURCE = f"{len(_y):,} val windows, {TEMPERATURE_N_CLASSES}-class"
    print(f"T refitted for {'+'.join(STREAMS)} on {TEMPERATURE_SOURCE}: {TEMPERATURE:.2f}")
else:
    TEMPERATURE = 1.0
    TEMPERATURE_N_CLASSES = None
    TEMPERATURE_SOURCE = "unfitted"
    print("WARNING: no val_logits.npz attached - serving UNCALIBRATED (T=1.0). Attach "
          "behaviorsense-results, or Viterbi and abstention consume meaningless posteriors.")
print(f"runs  {RUNS}")
print(f"qwen  {QWEN}")
print(f"rtmo  {RTMO or 'ABSENT - /video will refuse'}")
print(f"osnet {OSNET or 'ABSENT - roles will be UNKNOWN'}")
print(f"tau   {TAU}   T {TEMPERATURE}   auth {'on' if TOKEN else 'OFF (demo only)'}")
print()
print(f"  serving {len(STREAMS)} stream(s) from {RUNS}: {'+'.join(STREAMS)}")
print("  The 4-stream ensemble is the published headline; the served configuration is")
print("  the subset the lever table picked for macro-F1 (0.146 vs 0.128 for 4-stream).")
print("  See results/evaluation.md -> 'Accuracy levers' for the ablation. T is refit")
print("  for the served set above; tau swept with tau=0 asserted as an exact no-op.")

In [ ]:
# The API. Models load on a BACKGROUND thread so the tunnel address prints in seconds
# instead of after a three-minute weight load - /health reports `loading` until it is ready,
# which is what lets the front end say "models loading" rather than "offline".
import re, threading, time, numpy as np
from fastapi import FastAPI, HTTPException, Header
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from behaviorsense.agents.activity import ActivityConfig, FALLING, softmax, viterbi
from behaviorsense.eval.activity_eval import (apply_emergency_floor, class_prior,
                                              fit_transition_matrix, logit_adjust,
                                              sequences_by_subject)
from behaviorsense.models.ensemble import EnsembleClassifier, torch_supports_device
from behaviorsense.agents.reasoning.openrouter import FALLBACK_MODELS, OpenRouterLLM, discover_keys
# ONE subject decision for both halves of the split. It used to be inlined in this cell AND
# implemented in `staging.py`, which is two places for a rule about who the clip is about.
from behaviorsense.service.staging import assert_subject as _assert_subject

# Agent 4's hosted model. Pinned here rather than left to a default so the page's `model` field
# and this constant cannot disagree. GLM's OpenRouter page states it supports structured outputs
# via a JSON schema in `response_format`, which is what replaces `outlines`' local FSM.
OPENROUTER_MODEL = "z-ai/glm-5.2:free"
# THE FALLBACK CHAIN IS THE FIX, and it is a chain of PROVIDERS rather than of models.
# Measured 2026-08-28: thirteen keys returned thirteen 429s, one each, and OpenRouter's own body
# said `z-ai/glm-5.2:free is temporarily rate-limited UPSTREAM`; the dashboards showed several of
# those keys with no requests that day at all. The ceiling was Decart's capacity - one free
# endpoint shared by everyone - so no number of keys could help, because all of them arrive there.
# `FALLBACK_MODELS` is checked against `/api/v1/models/<id>/endpoints`: every entry supports strict
# `response_format` and every entry sits on a different provider. See that module for why
# `nemotron-3-ultra-550b-a55b:free` is not among them despite being the larger NVIDIA model.
OPENROUTER_FALLBACKS: tuple[str, ...] = FALLBACK_MODELS

app = FastAPI(title="BehaviorSense inference")
# The front end is served from a different origin by design (static host + Kaggle tunnel),
# so CORS is not optional. Methods are restricted; the origin cannot be, because the tunnel
# address is unknown until it is created.
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["GET", "POST"],
                   allow_headers=["*"])

STATE = {"ready": False, "error": None, "loaded_at": None}

def _load():
    try:
        import torch as _torch
        # Fragmentation, not capacity, is what the OOM message itself suggested trying
        # ("If reserved but unallocated memory is large try expandable_segments"). Set before
        # the first allocation or it has no effect.
        os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
        # Serve STREAMS, not everything on disk. from_run_dir() loads every adl_<stream>/
        # it finds, which is how the 4-stream ensemble became the default by accident.
        #
        # RESOLVED BY CONTENT, not by directory name. The Charades run wrote `adl_bone/`; the
        # Toyota RTMO run wrote `adl_toyota_bone/`, and a name-built path finds neither when
        # both are attached. Every checkpoint records its own `args.stream`, `args.n_classes`
        # and the shard paths it trained on, so the directory name is the least reliable thing
        # about it. ADL_PREFER picks between candidates for the same stream and is printed with
        # the alternatives, because with both datasets mounted the choice is real and silent
        # selection of the wrong corpus is the failure that matters.
        import torch as _t
        _cands = {}
        for _p in sorted(pathlib.Path(RUNS).glob("adl_*/best.pt")):
            try:
                _a = _t.load(str(_p), map_location="cpu", weights_only=False)
            except Exception as _e:                       # noqa: BLE001
                print(f"  skipped {_p.parent.name}: {type(_e).__name__}")
                continue
            _args, _m = _a.get("args", {}), _a.get("metrics", {})
            _cands.setdefault(_args.get("stream"), []).append({
                "path": _p, "dir": _p.parent.name,
                "n_classes": _args.get("n_classes"),
                "shards": " ".join(map(str, _args.get("shards", []))),
                "sample_fps": _args.get("sample_fps"),
                "mca": _m.get("mean_class_acc"), "f1": _m.get("macro_f1"),
            })
        print(f"  {len(_cands)} stream(s) with checkpoints under {RUNS}:")
        for _s, _lst in sorted(_cands.items(), key=lambda kv: str(kv[0])):
            for _c in _lst:
                print(f"    {_c['dir']:<26} stream={_s} classes={_c['n_classes']} "
                      f"mca={_c['mca']} f1={_c['f1']}")
        _ck, _absent = {}, []
        for _s in STREAMS:
            _lst = _cands.get(_s, [])
            _hit = [c for c in _lst if ADL_PREFER in c["dir"]] or _lst
            if not _hit:
                _absent.append(_s)
                continue
            if len(_hit) > 1:
                print(f"  WARNING {_s}: {len(_hit)} candidates match ADL_PREFER={ADL_PREFER!r} "
                      f"({[c['dir'] for c in _hit]}); taking the first. Narrow ADL_PREFER.")
            _ck[_s] = _hit[0]["path"]
            print(f"  serving {_s} from {_hit[0]['dir']} ({_hit[0]['n_classes']} classes)")
        assert not _absent, (
            f"no checkpoint for stream(s) {_absent} under {RUNS}. Found streams "
            f"{sorted(str(k) for k in _cands)}. Attach the runs dataset, or set STREAMS to what "
            "is actually trained.")
        # DOES THIS TORCH HAVE KERNELS FOR THIS CARD? `is_available()` says "there is a driver
        # and a device", not "this wheel was compiled for it". On Kaggle's P100 (sm_60) against a
        # torch built for sm_70+, the model moved to CUDA without complaint and Agent 2 then died
        # with `no kernel image is available for execution on the device` - which surfaced as an
        # AcceleratorError from whatever op ran first, reading like a bug in that op. RTMO was
        # unaffected because onnxruntime carries its own kernels.
        #
        # ST-GCN++ is 787k parameters over ~28 windows, so CPU costs a fraction of a second and is
        # the right answer rather than a degradation. P100 stays worthwhile: pose extraction is the
        # expensive stage, it is bandwidth-bound, and it ran 29.8 s there against 40 s on a T4.
        _torch_ok, _why = torch_supports_device()
        _clf_dev = "cuda" if _torch_ok else "cpu"
        print(f"  torch on GPU: {_torch_ok} - {_why}")
        if not _torch_ok:
            print(f"  ST-GCN++ and OSNet run on CPU (787k params over ~28 windows is "
                  f"sub-second); RTMO keeps the GPU through onnxruntime")
        clf = EnsembleClassifier(_ck, device=_clf_dev)
        STATE["torch_device"] = _clf_dev
        STATE["clf"] = clf
        STATE["streams"] = clf.streams
        STATE["n_classes"] = clf.n_classes

        # DOES THE TEMPERATURE BELONG TO THIS MODEL? `val_logits.npz` carries no link to the
        # checkpoint that produced it, so the config cell above can fit a temperature from any
        # file it finds. On the first served run it fitted T=0.79 from Charades logits (20
        # classes, 35,698 windows) onto this 22-class Toyota model - a calibration constant from
        # a different network, silently applied to the posteriors Viterbi and abstention consume.
        #
        # The head size is a decisive, free check, so it is made here where `clf` finally knows
        # its own. Discarding the fit costs calibration; keeping a foreign one corrupts every
        # smoothed label, and only one of those is recoverable.
        global TEMPERATURE
        if TEMPERATURE_N_CLASSES is not None and TEMPERATURE_N_CLASSES != clf.n_classes:
            print(f"  WARNING discarding T={TEMPERATURE:.2f}: it was fitted on "
                  f"{TEMPERATURE_SOURCE} logits but this checkpoint has {clf.n_classes} "
                  f"classes. Serving UNCALIBRATED (T=1.0). To calibrate, save val logits from "
                  f"the {clf.n_classes}-class run and attach those instead.")
            TEMPERATURE = 1.0
            STATE["temperature_source"] = "discarded (class-count mismatch)"
        else:
            STATE["temperature_source"] = TEMPERATURE_SOURCE
        STATE["temperature"] = TEMPERATURE
        # THE SAMPLING RATE COMES FROM THE CHECKPOINT'S OWN SHARDS, not from a constant here.
        # A 30-frame window is `30 / rate` seconds of motion and the model has seen exactly one
        # duration; Charades shards are 15 Hz (2.00 s) and the Toyota RTMO shards are 20 Hz
        # (1.50 s). Serving the Toyota weights at 15 Hz stretches every action by 1.33x, which
        # is the same defect as the old `src_fps / 2` stride wearing a different hat. Deriving it
        # from `args.shards` makes the pair impossible to separate.
        _sh = next((c["shards"] for l in _cands.values() for c in l
                    if c["path"] in _ck.values()), "")
        # `args.shards` is whatever the operator typed. Passed as a GLOB ("shards/*.npz") it
        # carries no rate at all, which is exactly what happened on the first served run - and
        # the fallback then quietly served 15 Hz weights trained at 20 Hz, a 1.33x stretch on
        # every action. So: expand the glob if the shards happen to be mounted, then try the
        # directory name, and REFUSE if neither answers.
        import glob as _glob
        _parts = [_sh]
        for _t in _sh.split():
            _parts.append(str(pathlib.Path(_t).parent))      # the shard DIRECTORY may name it
            if "*" in _t:
                _parts.extend(_glob.glob(_t))                 # expand, if they are mounted
        _hay = " ".join(_parts)
        # FIRST CHOICE: the rate the training run recorded. `train_adl.py` derives it from the
        # RESOLVED shard filenames, where the information actually exists - `args.shards` is
        # whatever was typed and a glob carries no rate at all, which is what made this refuse to
        # start once. Checkpoints trained before that fix fall through to the paths below.
        _rec = next((c.get("sample_fps") for l in _cands.values() for c in l
                     if c["path"] in _ck.values() and c.get("sample_fps")), None)
        _m = None if _rec else re.search(r"_(\d+(?:\.\d+)?)hz", _hay)
        if _rec:
            STATE["sample_fps"] = float(_rec)
            _why = "recorded by the training run"
        elif _m:
            STATE["sample_fps"] = float(_m.group(1))
            _why = "derived from the checkpoint's own shard paths"
        elif SAMPLE_FPS is not None:
            STATE["sample_fps"] = float(SAMPLE_FPS)
            _why = f"ASSERTED by SAMPLE_FPS={SAMPLE_FPS:g} (not derivable from args.shards)"
        else:
            # FAIL CLOSED. A wrong rate is invisible: the model returns confident labels for
            # windows of a duration it has never seen. Refusing costs a restart; guessing costs
            # every number the demo produces.
            raise AssertionError(
                "cannot determine the sampling rate this checkpoint was trained at. "
                f"args.shards = {_sh!r}. "
                "A 30-frame window is 30/rate seconds of motion and ST-GCN++ has seen exactly "
                "one duration; serving 15 Hz weights at 20 Hz - or the reverse - stretches every "
                "action by 1.33x and shows up only as bad accuracy. "
                "FIX: set SAMPLE_FPS in the config cell to the rate the shards were built at "
                "(15.0 for Charades, 20.0 for the Toyota RTMO shards), or re-run training with "
                "an expanded --shards list so the rate is recorded in args.shards.")
        print(f"  sample rate {STATE['sample_fps']:g} Hz "
              f"({30 / STATE['sample_fps']:.2f}s windows) {_why}")
        from behaviorsense.agents.reasoning.reporter import CaregiverReporter, ReporterConfig
        cfg = ReporterConfig(constrained=True)
        # SHARD ACROSS BOTH T4s. `device="cuda"` pins every layer to device 0, and Qwen2.5-7B
        # in 16-bit is ~15.2 GB of weights against a 14.56 GiB usable card - so it LOADED with
        # ~200 MB spare and then died on the first KV-cache allocation of generation:
        #
        #   OutOfMemoryError: Tried to allocate 34.00 MiB. GPU 0 has ... 10.81 MiB is free
        #
        # while `nvidia-smi` showed GPU 1 holding 3 MiB. The second card was never used. The
        # reporter has taken `max_memory` for exactly this since the T4 serving path was added;
        # this cell simply never passed it.
        #
        # The bounds are not tuning, they are reservations. Three other things want VRAM on
        # these same two cards:
        #   - the ST-GCN++ ensemble above, on device 0 (small, but it is already resident)
        #   - the RTMO + OSNet CHILD process, which opens its own CUDA context per upload
        #     (~2 GB with the onnxruntime session) and cannot share PyTorch's allocator
        #   - the KV cache, which grows during generation on whichever device holds the
        #     later layers - the allocation that actually failed
        # PyTorch's caching allocator does not return memory to the driver, so `auto` filling
        # both cards to the brim would starve the child even though the weights fit.
        _mm = {i: "11GiB" for i in range(_torch.cuda.device_count())} or None

        # DTYPE: float16 on a card WITHOUT native bfloat16, which is every serving card here.
        #
        # bf16 needs compute capability 8.0. T4 is sm_75 and P100 is sm_60, so a bf16 matmul does
        # not reach the tensor cores - it runs on a fallback path that is several times slower for
        # numerics nobody is measuring at serving time. That is most of why a 6-claim report took
        # 286 s with both GPUs reading 0% utilisation: emulated matmuls on top of pipeline-parallel
        # sharding across PCIe, at batch size 1.
        #
        # fp16 has a narrower exponent range and Qwen's weights are bf16-trained, so this is not
        # free - but `_assert_finite_logits()` runs one forward pass at load and refuses a
        # non-finite result, which is the failure mode fp16 actually has. The evaluation numbers
        # in results/evaluation.md were measured on the Blackwell in bf16 and are untouched by a
        # serving-side dtype.
        # From the CAPABILITY, not from `is_bf16_supported()`. That helper returned True on a
        # P100 whose torch build has no sm_60 kernels at all, and the loader duly printed
        # "reporter dtype bfloat16" for a card that cannot run bf16 or anything else. bf16 needs
        # sm_80; ask the device.
        _cap = _torch.cuda.get_device_capability(0) if _torch.cuda.is_available() else (0, 0)
        _bf16_native = _torch_ok and _cap >= (8, 0)
        _DTYPE = "bfloat16" if _bf16_native else "float16"
        print(f"  reporter dtype {_DTYPE}"
              + ("" if _bf16_native else
                 f" ({_torch.cuda.get_device_name(0)} is pre-sm_80, so bfloat16 would be "
                 "emulated off the tensor cores)"))
        # AGENT 4: HOSTED FIRST, LOCAL WEIGHTS AS THE FALLBACK.
        #
        # Qwen on two T4s was 286 s of a ~340 s request - 85% of the wall clock - for 17.8 GiB of
        # VRAM, emulated bf16 on sm_75, and ~13 GiB of host RAM retained per analysis. A hosted
        # endpoint at 164 tok/s makes that stage single-digit seconds and gives the pose model the
        # whole GPU. Skipping the local load also skips its 70 s and its footprint entirely.
        #
        # Safe for one structural reason, not because the hosted model is better: Agent 4 receives
        # NUMBERS ONLY - never frames, never skeletons - and every claim it writes goes through
        # C1-C5 arithmetically before anyone sees it. Substituting a model we know less about is
        # exactly the case the verifier exists to cover.
        #
        # Qwen remains the MEASURED arm. Every figure in results/evaluation.md came from it under
        # grammar-constrained decoding on a pinned checkpoint, and a `:free` endpoint can be
        # deprecated without notice. `model_name` travels in the payload and the page renders it,
        # so which model wrote a report is stated rather than assumed.
        _or_keys = discover_keys()
        if not _or_keys:
            try:
                from kaggle_secrets import UserSecretsClient   # noqa: PLC0415
                _sec = UserSecretsClient()
                # One secret holding a comma-separated list, because Kaggle secrets are one per
                # name and twenty of them is twenty clicks. Numbered names still work.
                for _n in ("OPENROUTER_API_KEY_LIST", "OPENROUTER_API_KEY"):
                    try:
                        _v = (_sec.get_secret(_n) or "").strip()
                    except Exception:                          # noqa: BLE001, S112
                        continue
                    if _v:
                        os.environ[_n] = _v
                _or_keys = discover_keys()
            except Exception as _e:                            # noqa: BLE001
                print(f"  kaggle_secrets unavailable ({type(_e).__name__}); "
                      "set OPENROUTER_API_KEY_LIST in Add-ons -> Secrets to use the hosted model")

        # NO LOCAL WEIGHTS ON THIS PATH. Qwen is gone from serving entirely: it was 286 s of a
        # ~340 s request on two T4s, and on the P100 it cannot run at all - torch has no sm_60
        # kernels, so it loaded for 106 s and then died with `no kernel image is available`
        # exactly as Agent 2 had. Agent 4 is a hosted call or it is not served.
        #
        # Qwen remains the MEASURED arm in notebook 04 on the Blackwell, which is where every
        # figure in results/evaluation.md came from and where a pinned checkpoint belongs.
        if _or_keys:
            # NOT cfg.max_new_tokens: that is the measured local arm's 1600, and a reasoning
            # model spends part of its budget thinking, so every hosted report came back as
            # `unterminated JSON object (truncated generation)`. See HOSTED_MAX_TOKENS.
            _llm = OpenRouterLLM(OPENROUTER_MODEL, keys=_or_keys,
                                 fallback_models=OPENROUTER_FALLBACKS)
            print(f"  agent 4: {_llm.name} - no local weights, no VRAM")
            print(f"    provider chain ({len(_llm.plan)}): {' -> '.join(_llm.plan)}")
            print("    a 429 saying 'rate-limited upstream' is the free PROVIDER saturated, not "
                  "your keys, so the chain changes provider rather than spending more keys on the "
                  "same one. If every provider is saturated at once, add a BYOK provider key at "
                  "openrouter.ai/settings/integrations for dedicated limits.")
        else:
            # Serving Agents 1-2 only is a WORKING configuration, not a failure. `/video?stages=12`
            # stops at the Agent 2 -> Agent 3 seam and the caller runs Agents 3-4 where its keys
            # already are - which is the point, since the keys need never reach Kaggle at all.
            _llm = None
            print("  agent 4: NOT SERVED HERE - no OpenRouter keys in this session.")
            print("    This is the intended split when your keys are local: run")
            print("      python web/local_backend.py --kaggle <this tunnel address>")
            print("    and point the front end at http://127.0.0.1:8899. It calls "
                  "/video?stages=12 here")
            print("    and runs Agents 3, 4 and C1-C5 on your machine, keys included.")
            print("    To serve Agent 4 from Kaggle instead, put a comma-separated list in")
            print("    Add-ons -> Secrets as OPENROUTER_API_KEY_LIST.")
        STATE["reporter"] = None if _llm is None else CaregiverReporter(_llm, config=cfg)
        STATE["agent4"] = None if _llm is None else _llm.name
        STATE["ready"] = True
        STATE["loaded_at"] = time.time()
    except Exception as exc:                      # noqa: BLE001 - surfaced via /health
        STATE["error"] = f"{type(exc).__name__}: {exc}"

threading.Thread(target=_load, daemon=True).start()

def _auth(tok):
    if TOKEN and tok != TOKEN:
        raise HTTPException(401, "bad or missing X-BS-Token")

@app.get("/health")
def health():
    import torch
    return {"service": "behaviorsense", "ready": STATE["ready"], "error": STATE["error"],
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "streams": STATE.get("streams", []), "tau": TAU, "auth": bool(TOKEN),
            # The front end hides the upload control when pose is unavailable rather than
            # offering one that 503s: a disabled button nobody can explain is worse than
            # a control that is honestly not there.
            # Which model writes the reports. The page shows it, because a hosted `:free`
            # endpoint must never be mistaken for the pinned checkpoint the measured numbers
            # came from.
            "agent4": STATE.get("agent4"),
            "video": bool(RTMO), "reid": bool(OSNET), "max_upload_mb": MAX_UPLOAD_MB,
            "max_frames": 900}

In [ ]:
# Three endpoints, mirroring the two halves of the system plus the one the front end needs.
#
# /demo exists because the alternative is worse: to exercise Agent 4 the browser would have
# to POST a fully-formed BehaviourState - DailyFeatures, thirty days of baselines, alerts -
# and hand-building that in JavaScript would mean the demo tests a payload nobody else
# constructs. Instead the backend runs the SAME simulator the evaluation uses, picks a day
# that actually alerts, and returns the claims, the verdicts, AND the evidence index they
# were checked against. The front end then re-runs its own port of the verifier over that
# evidence and compares - so the page shows two independent implementations agreeing on real
# model output, rather than asking anyone to trust one.
import json
from fastapi.responses import StreamingResponse
from behaviorsense.data.simulator import standard_scenarios

class Windows(BaseModel):
    windows: list          # [N, 30, 2, 17, 3] skeleton windows
    smooth: bool = True

class StatePayload(BaseModel):
    state: dict            # a serialised BehaviourState

@app.post("/activity")
def activity(body: Windows, x_bs_token: str = Header(default="")):
    _auth(x_bs_token)
    if not STATE["ready"]:
        raise HTTPException(503, STATE["error"] or "models still loading")
    X = np.asarray(body.windows, dtype=np.float32)
    if X.ndim != 5 or X.shape[-2:] != (17, 3):
        raise HTTPException(422, f"expected [N,T,M,17,3], got {list(X.shape)}")

    logits = STATE["clf"].logits(X)
    # Adjust BEFORE smoothing. Measured: smoothing the raw posterior cost 0.040 mean-class,
    # smoothing the adjusted one cost 0.028 and kept most of the top-1 gain.
    adjusted = logit_adjust(logits, class_prior(logits.argmax(1)), TAU)
    post = softmax(adjusted / 0.76)              # T fitted in notebook 04
    labels = post.argmax(1)
    if body.smooth:
        cfg = ActivityConfig()
        A = apply_emergency_floor(
            fit_transition_matrix([labels]), FALLING, cfg.emergency_floor)
        labels = viterbi(np.log(np.clip(post, 1e-12, None)), np.log(A),
                         np.log(np.full(post.shape[1], 1.0 / post.shape[1])))
    return {"labels": labels.tolist(), "confidence": post.max(1).round(4).tolist(),
            "tau": TAU, "smoothed": body.smooth}

def _blocking(fn, label, every=10.0):
    # Run a blocking call in a worker thread, emitting a heartbeat while it works.
    #
    # (Comments, not a docstring: this whole cell is one triple-quoted literal in
    # _generate.py and an inner triple quote terminates it early - the same trap the
    # _AdjustedClassifier note in the /video cell already describes.)
    #
    # A generator cannot yield from inside a blocking call, so streaming one line per AGENT
    # left the longest silence in the stream equal to the slowest stage: Agent 3's line went
    # out, then nothing at all for the whole of generation. The browser's silence watchdog
    # fired at 90 s and reported a dead session while the GPU was working perfectly normally.
    # Observed on a 5-second clip.
    #
    # Per-stage lines were never sufficient by themselves. The silence that matters is the one
    # INSIDE a stage, and only a second thread can interrupt it. `every` bounds that silence
    # regardless of how long the work takes, which is the property the watchdog needs: the
    # client can then treat silence as genuine failure rather than as a slow model.
    #
    # Yields ("beat", {...}) zero or more times, then exactly one ("done", value). Exceptions
    # propagate out of `fut.result()` on the caller's thread, so existing try/except still
    # catches them where it did before.
    import concurrent.futures as _cf
    with _cf.ThreadPoolExecutor(max_workers=1) as ex:
        fut = ex.submit(fn)
        t0 = time.time()
        while True:
            try:
                value = fut.result(timeout=every)
                break
            except _cf.TimeoutError:
                yield "beat", {"step": label, "elapsed_s": round(time.time() - t0, 1)}
    yield "done", value


def _serialise(out, state):
    from behaviorsense.agents.reasoning.verifier import FaithfulnessVerifier
    index = FaithfulnessVerifier().build_index(state)
    return {
        "day": state.report_day.isoformat(),
        "subject": state.subject_role.value,
        "summary": out.report.summary,
        "recommendation": out.report.recommendation,
        "escalate": out.report.escalate,
        "claims": [c.model_dump(mode="json") for c in out.report.claims],
        "verifications": [v.model_dump(mode="json") for v in out.report.verifications],
        "evidence": {k: v.model_dump(mode="json") for k, v in index.items()},
        "alerts": [{"kind": a.kind.value, "severity": a.severity.value} for a in state.alerts],
        "hallucination_rate": out.hallucination_rate,
        "emitted": out.n_emitted_claims,
        "schema_rejected": out.dropped_claims,
        "model": out.report.model_name,
    }

@app.get("/demo")
def demo(scenario: int = 0, x_bs_token: str = Header(default="")):
    # STREAMED for the same reason /video is. Measured 2026-08-25 against a live T4: this
    # endpoint was cut at 125.8 s with Cloudflare's 524 while the model was still generating,
    # twice, on two different sessions. The cap is on time-to-first-byte, so `accepted` goes
    # out immediately and `progress` lines keep the connection busy while Qwen runs.
    #
    # NDJSON, same envelope as /video:
    #   {"event":"accepted"} -> {"event":"progress",...}* -> {"event":"result","payload":{...}}
    #   or {"event":"error","detail":...}
    # `result.payload` is exactly what this endpoint returned before.
    _auth(x_bs_token)
    if not STATE["ready"]:
        raise HTTPException(503, STATE["error"] or "models still loading")
    # 503 with the reason, not a KeyError 500. This backend can legitimately serve Agents 1-2
    # only - on a GPU whose torch build has no kernels, there is nothing to write reports with.
    if STATE.get("reporter") is None:
        raise HTTPException(503, "this backend serves Agents 1-2 only: no reporter is configured. "
                                 "Set OPENROUTER_API_KEY_LIST, attach the Qwen model on a "
                                 "supported GPU, or use /video?stages=12 and run Agent 4 yourself.")
    scenarios = list(standard_scenarios())
    if not 0 <= scenario < len(scenarios):
        raise HTTPException(404, f"scenario {scenario} of {len(scenarios)}")

    def stream():
        def emit(obj):
            return json.dumps(obj, default=str) + "\n"

        key = f"demo:{scenario}"
        try:
            yield emit({"event": "accepted", "scenario": scenario,
                        "cached": key in STATE})
            if key in STATE:                  # a generation takes >2 min; cache per scenario
                yield emit({"event": "result", "payload": STATE[key]})
                return
            from behaviorsense.agents.behaviour import BehaviourAnalyzer
            analyzer = BehaviourAnalyzer()
            picked = None
            for day in scenarios[scenario].run().days:
                st = analyzer.analyze_day(day)
                if st.baselines and st.alerts:
                    picked = st               # last alerting day: the decline is developed
            if picked is None:
                yield emit({"event": "error", "kind": "simulator",
                            "detail": "simulator produced no alerting day"})
                return
            yield emit({"event": "progress", "step": "state_ready",
                        "day": picked.report_day.isoformat(),
                        "alerts": len(picked.alerts),
                        "note": "writing the report - this is the slow stage"})
            # HEARTBEAT THROUGH GENERATION. Without this the stream goes quiet for the whole
            # of `report()`, which is minutes on a T4, and the client cannot tell that from a
            # dead session.
            out = None
            for _kind, _obj in _blocking(lambda: STATE["reporter"].report(picked),
                                         "qwen_generating"):
                if _kind == "beat":
                    yield emit({"event": "heartbeat", **_obj})
                else:
                    out = _obj
            STATE[key] = _serialise(out, picked)
            yield emit({"event": "result", "payload": STATE[key]})
        except Exception as exc:                                   # noqa: BLE001
            import traceback
            yield emit({"event": "error", "kind": "server",
                        "detail": f"{type(exc).__name__}: {exc}",
                        "traceback": traceback.format_exc()[-1200:]})

    return StreamingResponse(stream(), media_type="application/x-ndjson",
                             headers={"Cache-Control": "no-store",
                                      "X-Accel-Buffering": "no"})

@app.post("/report")
def report(body: StatePayload, x_bs_token: str = Header(default="")):
    _auth(x_bs_token)
    if not STATE["ready"]:
        raise HTTPException(503, STATE["error"] or "models still loading")
    if STATE.get("reporter") is None:
        raise HTTPException(503, "no reporter is configured on this backend - it serves Agents "
                                 "1-2 only. Run Agent 4 where your API keys are.")
    from behaviorsense.schemas import BehaviourState
    state = BehaviourState(**body.state)
    return _serialise(STATE["reporter"].report(state), state)

In [ ]:
# /video - the only endpoint that touches pixels, and the only one that can be handed a
# file nobody vetted.
#
# Decode runs in a CHILD process (behaviorsense.video.extract_isolated). That is not
# defensive habit: ffmpeg raises SIGSEGV/SIGABRT on malformed streams, a signal is not an
# exception, and notebook 02 lost finished corpora to exactly this before it grew a
# journalling subprocess probe. Inline, one bad upload would kill this kernel and take the
# tunnel, the models and the demo with it. Behind the boundary it is a 422.
#
# What comes back is STAGED: one record per agent, in order, with its own timing, payload and
# status, plus the C1-C5 verdict on every individual claim. That shape exists so the front end
# can show what each agent produced and what the next one received, for one person or several.
#
# Agents 3 and 4 DO run on a single clip, which earlier versions refused. The refusal was
# right about the science and wrong about the remedy: Agent 3's baseline is a 14-day rolling
# median, so one clip cannot supply this person's history. Rather than omit the stages or
# fabricate a history, the baseline is a DECLARED simulated reference and every affected
# number says so. The feature values are measured from the video and are real; the robust-z
# and alerts are reference-relative. C1-C4 still check each claim against the state computed
# from this video, so the verifier's guarantee is unchanged - what the reference cannot
# support is the clinical reading, and the payload says that too.
import json, shutil, tempfile, threading, time, numpy as np
_VIDEO_LOCK = threading.Lock()


def _rss_mb():
    # Resident set size, from /proc - no dependency, exact, and the only number that answers
    # "which stage grows the process". RAM went 16.5 -> 29.6 GiB across two uploads on a 30 GiB
    # box, so a third would be killed. Guessing has already cost two wrong diagnoses; the
    # simulator rebuild, for one, is 1.3 MB measured and is not the cause.
    try:
        with open("/proc/self/status") as _f:
            for _l in _f:
                if _l.startswith("VmRSS:"):
                    return round(int(_l.split()[1]) / 1024.0, 1)
    except Exception:                                              # noqa: BLE001
        pass
    return None
from fastapi import File, Form, UploadFile
from fastapi.responses import StreamingResponse
from behaviorsense.agents.activity import (CLASS_NAMES, EXTENDED_CLASS_NAMES,
                                            FALLING, FALLEN)
def class_names_served():
    # The name list matching the head that is ACTUALLY loaded. Resolved per request.
    #
    # (Comments, not a docstring: this whole cell is one triple-quoted literal in _generate.py and
    # an inner triple quote terminates it early - the same trap _AdjustedClassifier documents.)
    #
    # Charades checkpoints emit 20 classes; the Toyota RTMO run emits 22, where ids 20-21 are
    # `using_device` and `object_interaction`. Indexing the 20-entry tuple at 20 raises
    # `IndexError: tuple index out of range`.
    #
    # This was a module-level constant, and that could not work: `_load()` runs on a BACKGROUND
    # thread so the tunnel address prints in seconds rather than after the weights load, so
    # STATE has no `n_classes` when this cell executes. The `.get()` default won every time and the
    # 20-name tuple was captured permanently - for a 22-class model. It surfaced in the top-3
    # diagnostic, but the same tuple names every segment, so any clip Agent 2 labelled
    # `using_device` or `object_interaction` would have 500'd the whole upload.
    #
    # Read from STATE at call time, falling back to the classifier itself rather than to a length:
    # a wrong name is worse than a missing one, because it is reported as fact.
    n = STATE.get("n_classes")
    if n is None:
        clf = STATE.get("clf")
        n = getattr(clf, "n_classes", None) or len(CLASS_NAMES)
    return EXTENDED_CLASS_NAMES if int(n) > len(CLASS_NAMES) else CLASS_NAMES
from behaviorsense.agents.activity import ActivityConfig
from behaviorsense.pipeline import (ActivityPipeline, frames_to_windows,
                                   observed_hours_from_frames)
from behaviorsense.data.skeleton_dataset import normalise as _normalise
from behaviorsense.schemas import Role
from behaviorsense.video import MAX_FRAMES, extract_isolated, rebuild_observations

@app.post("/video")
async def video(file: UploadFile = File(...), x_bs_token: str = Header(default=""),
                stages: str = "1234",
                # The operator's gallery, as JSON, and an optional enrol name. Both arrive
                # from the LOCAL backend (web/local_backend.py), which owns gallery.json -
                # the file never lives on Kaggle, so biometric templates never touch a
                # third-party service. Direct-to-tunnel uploads simply omit them.
                gallery: str = Form(""),
                enrol: str = Form("")):
    _auth(x_bs_token)
    if not STATE["ready"]:
        raise HTTPException(503, STATE["error"] or "models still loading")
    if not RTMO:
        raise HTTPException(503, "rtmo-l.onnx is not attached, so pose extraction is off")

    # The upload is read HERE, while an HTTP status is still available to reject it. Once the
    # streaming response has begun, headers are sent and 413/422 are no longer options.
    td = tempfile.mkdtemp(prefix="bs-upload-")
    # The operator's gallery, materialised as a file for the decode child. Written under the
    # upload's own temp dir so the existing finally-rmtree removes it - biometric templates
    # must not outlive the request on this machine, which is the whole reason the gallery
    # lives on the operator's side and travels per request.
    use_osnet = OSNET and bool((gallery or "").strip() or enrol.strip())
    gallery_path = None
    if use_osnet and gallery.strip():
        gallery_path = pathlib.Path(td) / "gallery.json"
        gallery_path.write_text(gallery.strip(), encoding="utf-8")
    dst = pathlib.Path(td) / (pathlib.Path(file.filename or "clip").name or "clip")
    size = 0
    try:
        with dst.open("wb") as fh:
            # Streamed in chunks and capped as it arrives. Reading the whole body first to
            # measure it would let a large upload exhaust memory before the check runs.
            while chunk := await file.read(1 << 20):
                size += len(chunk)
                if size > MAX_UPLOAD_MB << 20:
                    raise HTTPException(413, f"upload exceeds {MAX_UPLOAD_MB} MB")
                fh.write(chunk)
        if not size:
            raise HTTPException(422, "empty upload")
    except BaseException:
        shutil.rmtree(td, ignore_errors=True)
        raise

    # ONE CLIP AT A TIME. Two concurrent uploads each spawn a decode child holding its own CUDA
    # context for RTMO and OSNet, and both then contend for a Qwen generation that already fills
    # both T4s - the kernel died rather than either finishing. A 429 with a plain reason is a
    # worse demo than a queue and a far better one than a dead tunnel, and the front end can act
    # on it. `_VIDEO_LOCK` is non-blocking on purpose: queueing behind a 5-minute request would
    # be indistinguishable from a hang.
    if not _VIDEO_LOCK.acquire(blocking=False):
        raise HTTPException(429, "a clip is already being analysed - this session runs one at a "
                                 "time because pose extraction and the 7B model each need the "
                                 "whole GPU. Wait for the current run to finish and retry.")
    try:
            # `?stages=12` stops after Agent 2 and returns poses + segments only. That is the
        # AGENT 2 -> AGENT 3 seam, which is where this project already claims pixels stop, so a
        # caller can run Agents 3 and 4 wherever their LLM keys live - off this box, out of Kaggle
        # Secrets, and off the GPU. The full "1234" default keeps standalone Kaggle working.
        return StreamingResponse(_video_stream(td, dst, time.time(), stages=stages,
                                              use_osnet=use_osnet,
                                              gallery_path=gallery_path),
                                 media_type="application/x-ndjson",
                                 headers={"Cache-Control": "no-store",
                                          "X-Accel-Buffering": "no"})
    except BaseException:
        _VIDEO_LOCK.release()
        raise


def _video_stream(td, dst, t_start, stages="1234", use_osnet=False, gallery_path=None):
    # STREAMED, one line per agent, and this is a correctness fix rather than a nicety.
    #
    # The quick tunnel drops a request whose origin has not answered in about two minutes:
    # measured 2026-08-25, /demo was cut at 125.8 s with Cloudflare's 524 while the T4 was
    # still generating. /video is strictly slower - RTMO over up to 900 frames, then
    # ST-GCN++, then that same Qwen pass - so one synchronous JSON response cannot carry it.
    #
    # Cloudflare's limit is on time-to-first-byte, so the fix is to send a byte at once and
    # keep sending. Shortening the report or lowering max_frames would trade measured
    # behaviour for a timeout, which is the trade this project keeps refusing.
    #
    # Newline-delimited JSON. `stage` arrives as each agent finishes, which is also exactly
    # what the front end wants to draw, so the timeout fix and the feature are the same code:
    #   {"event":"accepted"}              first, immediately - stops the 524 clock
    #   {"event":"stage","stage":{...}}   one per agent, in order, as it completes
    #   {"event":"heartbeat",...}         inside a long stage, so silence stays bounded
    #   {"event":"result","payload":{...}} the SAME dict this endpoint used to return
    #   {"event":"error","detail":...}    IN-BAND: headers are already sent by now, so an
    #                                    HTTP status can no longer carry a failure
    # `result` is byte-for-byte the old payload, so stages.js and the contract tests keep
    # describing one shape.
    def emit(obj):
        return json.dumps(obj, default=str) + "\n"

    try:
        yield emit({"event": "accepted", "max_frames": MAX_FRAMES,
                    "note": "decoding in a child process; stages follow as they finish"})
        try:
            # Pose runs in a child process over up to MAX_FRAMES frames. It is minutes of GPU on a
            # long upload, and the child itself only speaks every PROGRESS_EVERY frames - so
            # `_blocking` emits a heartbeat on its own 10 s timer regardless of what the child
            # reports. That timer, not the child, is what keeps the browser's silence watchdog fed.
            poses = None
            for _kind, _obj in _blocking(
                    lambda: extract_isolated(str(dst), rtmo=RTMO, device="cuda",
                                             # RTMO on CUDA via onnxruntime; OSNet follows torch's
                                             # own arch list, which is a different question - on a
                                             # P100 the ONNX model runs and every torch kernel
                                             # fails with "no kernel image is available".
                                             osnet_device=STATE.get("torch_device", "cuda"),
                                             # OSNet only when this request can use it: a
                                             # gallery to match against, or an enrolment to
                                             # collect crops for. Otherwise the 104 s CPU
                                             # cost buys the string "unidentified".
                                             osnet=OSNET if use_osnet else None,
                                             # The operator's gallery, as a file under this
                                             # request's temp dir. `extract_isolated` passes
                                             # it by PATH so biometric data never appears in
                                             # a process listing, and the existing
                                             # finally-rmtree removes it with the upload.
                                             gallery=gallery_path,
                                             target_fps=STATE.get("sample_fps", SAMPLE_FPS),
                                             # NO TIMEOUT. `extract_isolated` watches the child's
                                             # own progress lines instead and gives up only on
                                             # silence, because any wall-clock ceiling has to be
                                             # set high enough for the longest acceptable video and
                                             # so cannot tell a slow clip from a wedged GPU.
                                             max_frames=MAX_FRAMES),
                    "rtmo_decoding"):
                if _kind == "beat":
                    yield emit({"event": "heartbeat", **_obj})
                else:
                    poses = _obj
        except ValueError as exc:
            yield emit({"event": "error", "kind": "decode", "detail": str(exc)})
            return
        finally:
            shutil.rmtree(td, ignore_errors=True)

        t_pose = time.time()
        for kind, obj in _video_stages(poses, t_start, t_pose, stages=stages,
                                       reid_note={"requested": bool(use_osnet),
                                                  "gallery_attached": gallery_path is not None,
                                                  "weights_present": bool(OSNET)}):
            if kind == "stage":
                yield emit({"event": "stage", "stage": obj})
            elif kind == "heartbeat":
                yield emit({"event": "heartbeat", **obj})
            else:
                yield emit({"event": "result", "payload": obj})
    except Exception as exc:                                       # noqa: BLE001
        # In-band, because the response has already begun. A truncated NDJSON stream with no
        # error line is indistinguishable to the client from a dropped connection, and this
        # project has spent enough sessions on failures that looked like something else.
        import traceback
        yield emit({"event": "error", "kind": "server",
                    "detail": f"{type(exc).__name__}: {exc}",
                    "traceback": traceback.format_exc()[-1200:]})
    finally:
        shutil.rmtree(td, ignore_errors=True)
        # RECLAIM BEFORE THE NEXT UPLOAD. Two runs took RSS from 16.5 to 29.6 GiB of 30, so a
        # third is killed by the OOM reaper - and a killed kernel takes the tunnel with it. This
        # does not fix a leak, it buys the demo a third run: `gc.collect()` frees the reference
        # cycles a generator's frames leave behind, and `empty_cache()` returns the allocator's
        # unused blocks so the decode child can open its CUDA context next time.
        try:
            import gc as _gc, torch as _tt
            _freed = _gc.collect()
            if _tt.cuda.is_available():
                _tt.cuda.empty_cache()
            print(f"  reclaimed {_freed} objects; RSS now {_rss_mb()} MB")
        except Exception as _e:                                    # noqa: BLE001
            print(f"  reclaim skipped: {type(_e).__name__}: {_e}")
        # Released HERE, not where the response was constructed: the work happens while this
        # generator is consumed, so releasing earlier would let a second upload in mid-run.
        _VIDEO_LOCK.release()


def _video_stages(poses, t_start, t_pose, stages="1234", reid_note=None):
    _rss = {"start": _rss_mb()}
    # The four agents over one clip, yielding each stage record as it is finished rather than
    # after all of them are. Transport lives in `_video_stream`; this function knows nothing
    # about HTTP, which is what lets the contract suite execute it on a laptop.
    frames = rebuild_observations(poses)

    def _stage(n, name, status, elapsed, payload):
        return {"agent": n, "name": name, "status": status,
                "elapsed_s": round(elapsed, 2), "payload": payload}

    # AGENT 1, emitted NOW - before Agent 2 has run, which is the point of streaming.
    #
    # `n_people` counts distinct track ids in AGENT 1's own output, not in Agent 2's segment
    # list. Those differ, and the difference is the interesting case: a person who is tracked
    # but too occluded to classify produces no segments, so counting Agent 2's tracks reported
    # fewer people than the overlay draws. The page documents that state as a dashed
    # "pose unusable" box - "present, unreadable" and "not there" are different facts, and
    # Agent 1's own card must report the one Agent 1 established.
    tracked = {}
    _seen_roles = {}
    for _f in frames:
        for _p in _f.persons:
            # THE SETTLED ROLE, not the first frame's. `setdefault` kept the FIRST
            # observation, and the first observation of every track is `unknown` by
            # construction - the role vote has no evidence yet (see P3d's "honest ramp-up").
            # So Agent 1's card said `unidentified · track 0` for a track Agent 2 reported on
            # the same page as `resident ... Mary`. Two cards disagreeing about identity is
            # worse than either answer alone. Majority over the non-unknown observations: the
            # ramp-up frames do not outvote what the track actually converged to.
            _seen_roles.setdefault(_p.track_id, []).append(_p.role.value)
    # Resolve BEFORE stage1 reads `tracked` - it reports `n_people` as len(tracked), so a
    # resolution placed later left Agent 1 announcing zero people on a clip with two.
    for _tid, _roles in _seen_roles.items():
        _known = [r for r in _roles if r != "unknown"]
        tracked[_tid] = max(set(_known), key=_known.count) if _known else "unknown"

    stage1 = _stage(1, "perception", "done", t_pose - t_start, {
        "fps": poses["fps"], "frames_kept": poses["frames_kept"],
        "truncated": poses["truncated"], "providers": poses["providers"],
        "n_people": len(tracked), "reid": poses["reid"],
        # WHY re-identification did or did not run. Without this, "re-id off" on a request
        # that asked to enrol is indistinguishable from a request that did not ask - which
        # is exactly the ambiguity a failed first enrolment produced.
        "reid_note": reid_note or {},
        # Pose QUALITY, so a bad overlay can be diagnosed from the page instead of guessed at.
        # RTMO has no detector in front of it: on a cluttered scene it invents low-confidence
        # people, and it also loses real ones whose joints all fall under the threshold. Those
        # two look the same on screen, and both look like "the model is broken".
        "mean_kp_score": poses.get("mean_kp_score"),
        "weak_person_records": poses.get("weak_person_records"),
        "person_records": poses.get("person_records"),
        "container_size": poses.get("container_size"),
        "decoded_size": [poses.get("width"), poses.get("height")],
        # Temporal geometry. A 30-frame window is 2.0 s at the 15 Hz the shards were built at,
        # and ST-GCN++ has seen no other duration. Reported so a rate mismatch is visible
        # rather than absorbed as bad accuracy.
        "source_fps": poses.get("source_fps"),
        "window_seconds": poses.get("window_seconds"),
        # Against the SERVED rate, not `video.SHARD_FPS`. That constant is 15.0 (Charades), so a
        # correctly-configured 20 Hz Toyota run was told its labels were outside measured
        # conditions - the banner accusing the one configuration that is right.
        "rate_matches_shards": abs(poses["fps"] - STATE.get("sample_fps", SAMPLE_FPS)) < 0.01,
        "trained_fps": STATE.get("sample_fps", SAMPLE_FPS),
        "tracks": [{"track_id": k, "role": v} for k, v in sorted(tracked.items())],
    })
    _rss["after_pose"] = _rss_mb()
    yield "stage", stage1

    # AGENT 2 THROUGH THE LIBRARY PATH, not a re-implementation. This endpoint used to do
    # its own logit_adjust -> softmax -> argmax -> run-length pass, which SKIPPED Viterbi
    # smoothing and abstention entirely - so the served labels were not the configuration
    # notebook 04 evaluated, and the demo would have shown numbers no table backs. The
    # adapter applies the selected tau inside `logits()` so `ActivityPipeline` sees exactly
    # the posteriors the measured configuration produces.
    class _AdjustedClassifier:
        # The served ensemble with the selected logit adjustment folded in. A `"""`
        # docstring cannot go here: this whole cell is one triple-quoted literal in
        # _generate.py, and an inner triple quote terminates it early - which is exactly
        # how this cell broke the generator once already.
        def __init__(self, inner, tau):
            self.inner, self.tau = inner, tau
            # FORWARD the head size. `ActivityPipeline` reads `n_classes` off the classifier to
            # size the transition matrix and the Viterbi prior; an adapter that swallows the
            # attribute silently reverts the decoder to 20 classes, and the mismatch surfaces as
            # `operands could not be broadcast together with shapes (20,) (22,)` inside Viterbi -
            # three stages after the wrapper that caused it.
            _n = getattr(inner, "n_classes", None)
            if _n:
                self.n_classes = int(_n)

        def logits(self, windows):
            lg = self.inner.logits(windows)
            # n_classes FROM THE LOGITS. `class_prior` defaults to 20, so a 22-class
            # head produced a (20,) prior against (N,22) logits and raised
            # "operands could not be broadcast together with shapes (28,22) (1,20)" -
            # in Agent 2, after Agent 1 had already spent 35 s on pose.
            return logit_adjust(
                lg, class_prior(lg.argmax(1), n_classes=lg.shape[1]), self.tau)

    pipe = ActivityPipeline(_AdjustedClassifier(STATE["clf"], TAU),
                            ActivityConfig(temperature=TEMPERATURE))
    obs_hours = observed_hours_from_frames(frames, fps=poses["fps"] or 15.0)
    segments = pipe.run(frames)

    # WHO IS THE SUBJECT? On an uploaded clip, nobody - and that silently zeroed everything.
    #
    # `aggregate_daily_features` attributes personal features only to segments whose role IS
    # the subject role: `subject = [s for s in segments if s.role is subject_role]`. OSNet
    # decides that role by matching against an ENROLLED gallery, and a stranger's clip has no
    # gallery, so every track comes back `unknown`, `subject` is empty, and all 27 features are
    # 0.0 - including `social_interaction_duration_s` while Agent 2 was reporting 4.4 s of
    # `interacting_with_person`. Deviations then clamp at -10 on every feature and Agent 4
    # writes a confident report about a person who did nothing. Observed, on a 5-second clip of
    # someone washing dishes.
    #
    # Refusing to run is not better - it would hide the verifier, which is the part worth
    # showing. So the clip's most-present track is treated as the subject and that assumption
    # is DECLARED, exactly as the simulated baseline already is. Identity here is ASSERTED,
    # not re-identified, and `subject_provenance` says so on the payload and in the banner.
    #
    # A real ReID match is never overridden: if any segment already carries RESIDENT, the
    # gallery spoke and this stays out of the way.
    _seg_roles = {s.role for s in segments}      # noqa: F841 - kept for the assertion below
    # Captured BEFORE any relabel. The cards must report what Agent 1's re-identifier actually
    # decided; showing "resident" where OSNet said `unknown` would be the system lying about
    # its own identity layer to make the demo tidier.
    reid_roles = {s.track_id: s.role.value for s in segments}
    subject_track, subject_provenance = None, "reid_matched"
    subject_tracks: list[int] = []
    # HOISTED out of the branch below: a caller running Agents 3-4 itself needs these counts to
    # assert a subject, and they were only built when re-identification had already failed - so
    # the seam return NameError'd on exactly the clips where the gallery did match.
    _frames_per_track = {}
    _spans: dict[int, tuple[int, int]] = {}
    _boxes: dict[int, tuple[list[float], list[float]]] = {}
    for _f in frames:
        for _p in _f.persons:
            _frames_per_track[_p.track_id] = _frames_per_track.get(_p.track_id, 0) + 1
            _i = int(_f.frame_idx)
            _cur = _spans.get(_p.track_id)
            _spans[_p.track_id] = (_i, _i) if _cur is None else (min(_cur[0], _i),
                                                                max(_cur[1], _i))
            _box = [float(_p.box.x1), float(_p.box.y1),
                    float(_p.box.x2), float(_p.box.y2)]
            _seen = _boxes.get(_p.track_id)
            # First and last box only: the merge asks whether a person could have WALKED from
            # where one fragment ended to where the next began, so the ends are the whole question.
            _boxes[_p.track_id] = ((_box, _box) if _seen is None else (_seen[0], _box))
    # ONE implementation of the subject decision, shared with `web/local_backend.py`. This used to
    # be inlined here, which is how the two halves could disagree about who the clip is about; the
    # merge rule is subtle enough that two copies of it is not a risk worth taking.
    segments, subject_track, subject_provenance, subject_tracks = _assert_subject(
        segments, _frames_per_track, spans=_spans, boxes=_boxes,
        fps=float(poses["fps"] or SAMPLE_FPS))

    # Resolved ONCE per request, from the classifier that is actually loaded. Both the segment
    # names and the diagnostic below index it, and a stale 20-entry tuple raises on exactly the two
    # classes Toyota added.
    _names = class_names_served()
    if len(_names) < (STATE.get("n_classes") or len(_names)):
        raise AssertionError(
            f"the served head has {STATE.get('n_classes')} classes but only {len(_names)} names "
            "are available - naming a class by the wrong string would report a wrong activity as "
            "fact. Extend EXTENDED_CLASS_NAMES to match the taxonomy the checkpoint was trained "
            "on.")

    by_track = {}
    for s in segments:
        # `s.role`, not `s.subject_role`. ActivitySegment carries the role of the ONE person
        # it belongs to; `subject_role` is a BehaviourState field (the day's subject). Getting
        # this wrong 500'd every upload, and it 500'd OUTSIDE the try/except below, so it took
        # the whole request rather than degrading to a failed stage.
        rec = by_track.setdefault(s.track_id, {
            "track_id": s.track_id,
            # The RE-IDENTIFIED role, not the possibly-asserted one. `is_subject` carries the
            # assertion separately so the page can say "this track was treated as the subject"
            # without claiming OSNet recognised anybody.
            "role": reid_roles.get(s.track_id, s.role.value),
            "is_subject": s.track_id == subject_track,
            # The enrolled NAME the gallery matched this track to, when it matched. The role
            # already says resident/stranger; the name is what a caregiver reads.
            "matched_name": (poses.get("track_names") or {}).get(str(s.track_id)),
            "segments": []})
        rec["segments"].append({
            "label": int(s.activity_id), "name": _names[int(s.activity_id)],
            "t0": round((s.start_time - frames[0].timestamp).total_seconds(), 2),
            "t1": round((s.end_time - frames[0].timestamp).total_seconds(), 2),
            "confidence": round(float(s.confidence), 3),
            "room": s.room,
        })
    tracks = list(by_track.values())
    for t in tracks:
        t["n_segments"] = len(t["segments"])
        t["fall_segments"] = sum(1 for q in t["segments"] if q["label"] in (FALLING, FALLEN))
    t_cls = time.time()

    # AGENT 2, emitted as soon as it has finished - the front end draws the timelines while
    # Agent 3 is still priming and Qwen has not been called.
    # WHAT WAS AGENT 2 CHOOSING BETWEEN? A single segment spanning the whole clip is either a
    # confident correct call or Viterbi's self-transition prior (0.9) collapsing 28 uncalibrated
    # windows into one state, and the segment list alone cannot tell those apart. The top-3
    # per-window posteriors make the difference visible instead of arguable.
    _top = []
    try:
        # NORMALISE HERE TOO. This diagnostic bypassed `ActivityPipeline` to reach the raw
        # windows and therefore reproduced the exact bug it exists to diagnose - the guard
        # refused it with "windows peak at 356 in x/y". Written to explain a scale error and
        # containing one.
        _w = np.stack([_normalise(w.astype(np.float32))
                       for w in next(iter(frames_to_windows(frames))).windows])
        _lg = STATE["clf"].logits(_w)
        _pp = softmax(logit_adjust(_lg, class_prior(_lg.argmax(1), n_classes=_lg.shape[1]), TAU)
                      / max(TEMPERATURE, 1e-6))
        for _row in _pp[:40]:
            _o = np.argsort(-_row)[:3]
            _top.append([[_names[int(i)], round(float(_row[int(i)]), 3)] for i in _o])
    except Exception as _e:                                        # noqa: BLE001
        _top = [[["diagnostic unavailable", 0.0]]]
        print(f"  top-3 diagnostic skipped: {type(_e).__name__}: {_e}")

    stage2 = _stage(2, "activity", "done" if tracks else "skipped", t_cls - t_pose, {
        "window_top3": _top,
        "n_windows": len(_top),
        "streams_served": STATE.get("streams"), "tau": TAU, "temperature": TEMPERATURE,
        "decoding": "calibrate -> object fusion -> Viterbi -> abstain -> segments",
        "n_segments": len(segments),
        "per_track": tracks,
        "subject_provenance": subject_provenance,
        "subject_track": subject_track,
        # ALL the tracks attributed to the subject, not just the primary. One person fragmented
        # into three ids on a 45 s clip, and naming only the most-present one made the page claim
        # a single track while Agent 3 counted a merged set.
        "subject_tracks": list(subject_tracks),
    })
    _rss["after_classify"] = _rss_mb()
    yield "stage", stage2

    # AGENTS 3 AND 4 ON ONE CLIP - and why this needs a DECLARED provenance, not a quiet
    # default. Agent 3's deviation is robust-z against a 14-day rolling median. One upload is
    # a single moment, so there is no history for this person. Two options existed and only
    # one is honest: fabricate a baseline and let the report read as though it were checked
    # against the resident's own past, or run the layer against a declared reference and say
    # so on every affected number.
    #
    # The verifier is unaffected either way, and that is the point. C1-C4 check each claim
    # against the state actually computed from THIS video, so an invented figure, a wrong
    # percentage or an inverted direction is still caught arithmetically. What a reference
    # baseline cannot support is the clinical reading ("mobility declined"), because the
    # comparison point is not this person. Both facts ship in the response.
    stage3, stage4, checks = None, None, []
    t3 = t_cls
    st = None
    if "3" not in stages:
        # STOP AT THE SEAM. The caller runs Agents 3 and 4 themselves - typically a local backend
        # holding the LLM keys, so no credential ever reaches Kaggle Secrets or a saved notebook.
        # `stages` is echoed in the payload so a consumer cannot mistake a truncated run for a
        # complete one that produced no claims.
        yield "result", {
            "fps": poses["fps"], "width": poses["width"], "height": poses["height"],
            "frames_kept": poses["frames_kept"], "truncated": poses["truncated"],
            "reid": poses["reid"], "providers": poses["providers"],
            "frames": poses["frames"], "tracks": tracks, "n_people": len(tracked),
            "stages": [stage1, stage2], "checks": [], "stages_run": "12",
            "subject_provenance": subject_provenance, "subject_track": subject_track,
            "subject_tracks": list(subject_tracks),
            "frames_per_track": {str(k): v for k, v in _frames_per_track.items()},
            # Per-track first/last frame. Only this half has the per-frame observations, and the
            # local half needs them to reunite one person's fragmented ids - a track that leaves
            # detection for longer than the tracker's ~15 s bridge comes back with a new id.
            "track_spans": {str(k): [v[0], v[1]] for k, v in _spans.items()},
            # ENROLMENT MATERIAL for the local half: the subject track's best crops as
            # embeddings (never pixels), plus whether a gallery actually matched this clip -
            # the local backend persists these only when the operator asked to enrol.
            "gallery_updates": {
                "subject_track": subject_track,
                "matched": subject_provenance == "reid_matched",
                "embeddings": (poses.get("enrolment") or {}).get(str(subject_track), []),
            },
            # First and last box per track. The local half needs these to reject a fragment the
            # person could not have walked to - see `reachable` in service/staging.py. Without them
            # a television in shot merges into the resident, because a screen never shares frames
            # with the room and so looks exactly like a person who stepped out of detection.
            "track_boxes": {str(k): [v[0], v[1]] for k, v in _boxes.items()},
            "fps": poses["fps"],
            "day": frames[0].timestamp.date().isoformat(),
            "observed_hours": round(obs_hours, 6),
            "timing": {"pose_s": round(t_pose - t_start, 1),
                       "classify_s": round(t_cls - t_pose, 1),
                       "behaviour_s": 0.0, "report_s": 0.0},
            "rss_mb": _rss, "tau": TAU, "temperature": TEMPERATURE,
            "report": None, "report_provenance": None,
        }
        return
    if tracks:
        # SEPARATE try blocks per agent, deliberately. One block around both meant a Qwen
        # exception left `stage4` as None, which the stage record then reported as
        # "no reporter loaded (Qwen weights absent) or no track found" - blaming missing
        # weights for a crash in weights that had loaded. A failure must be attributed to the
        # agent that produced it, or the log sends the next person to the wrong place.
        try:
            from behaviorsense.agents.behaviour import BehaviourAnalyzer, aggregate_daily_features
            from behaviorsense.data.simulator import standard_scenarios
            from datetime import timedelta

            analyzer = BehaviourAnalyzer()
            ref = next(iter(standard_scenarios())).run()
            primed = 0
            for day in ref.days[:21]:
                analyzer.analyze_day(day)
                primed += 1
            # AGGREGATE THE SEGMENTS WE ALREADY HAVE. `pipe.run_to_features(frames)` calls
            # `self.run(frames)` internally, so it classified every window a SECOND time -
            # doubling the ST-GCN++ pass for nothing, and worse, discarding the subject relabel
            # above because its segments were a fresh set. Two independent runs also means
            # Agent 2's card and Agent 3's figures could disagree, which is the one thing a
            # stage-by-stage view must never do.
            today = aggregate_daily_features(
                segments, day=frames[0].timestamp.date(), observed_hours=obs_hours)
            # DATE THE CLIP AFTER THE REFERENCE WINDOW, or the whole layer comes back empty.
            #
            # Baselines are computed with `before=day_features.day`, and deviations exist only
            # for features that HAVE a baseline. An uploaded clip's timestamps start at the
            # decoder's own origin, which is 2026-01-01 - the same date the simulator's first
            # reference day carries. `before=2026-01-01` matches nothing, so baselines were
            # empty, so deviations were empty, so Agent 4 wrote "0 features reviewed" and
            # emitted no claims, so the C1-C5 table rendered blank on every upload. Every
            # stage still reported `done`, which is why this looked like it worked.
            today = today.model_copy(update={
                "day": ref.days[primed - 1].day + timedelta(days=1)})
            st = analyzer.analyze_day(today)
            stage3 = {
                "baseline_provenance": "reference_cohort_simulated",
                "subject_provenance": subject_provenance,
                "subject_track": subject_track,
                "subject_tracks": list(subject_tracks),
                "baseline_days": primed,
                "real_days_from_this_video": 1,
                "observed_hours": round(obs_hours, 3),
                # CALL IT. `is_reliable` is a METHOD, so `bool(getattr(...))` evaluates a bound
                # method - always truthy - and a 29-second clip (observed_hours 0.01) was reported
                # as a reliable day. `stages.js` suppresses its own "far below the 8 h a day needs
                # to be called reliable" warning on this field, so the bug deleted the one sentence
                # telling a caregiver not to act on the numbers.
                "is_reliable": bool(today.is_reliable()),
                # THE SAME GATE THE PROMPT USES, for the card. A ten-minute total against a
                # 22.9 h baseline median is a unit error (the clip's walking RATE was normal
                # while the raw z read -10.00), so the deviation table is withheld on a partial
                # window exactly as the numbers are withheld from Agent 4, and the page says why.
                "deviations_withheld": not bool(today.is_reliable()),
                "features_from_video": {k: round(float(v), 2)
                                        for k, v in today.numeric_items().items()},
                "deviations": ([] if not today.is_reliable() else
                               [{"feature": k, "robust_z": round(float(v), 2)}
                                for k, v in sorted(st.deviations.items(),
                                                   key=lambda kv: -abs(kv[1]))[:10]]),
                "alerts": [{"kind": a.kind.value, "severity": a.severity.value,
                            "rule": a.rule_name} for a in st.alerts],
                "caveats": [
                    "The BASELINE is a simulated reference persona, not this person's "
                    "history - one clip cannot supply 14 days.",
                    "Feature VALUES are measured from the uploaded video and are real.",
                    "Robust-z and alerts are therefore reference-relative: read them as "
                    "'unlike the reference', never as 'this person has declined'.",
                ] + ([
                    "IDENTITY IS ASSERTED, not re-identified: no resident is enrolled, so "
                    f"the most-present track ({subject_track}) was treated as the subject. "
                    "OSNet reports every track as unidentified, which is correct - it has no "
                    "gallery to match against. Without this the subject filter matches nothing "
                    "and all 27 features read 0.",
                ] if subject_provenance == "asserted_most_present_track" else []) + ([
                    f"TRACKS {', '.join(str(t) for t in subject_tracks)} WERE MERGED into one "
                    "subject because their frame spans never overlap, so they cannot be two "
                    "people present at the same time. A person who leaves detection for more "
                    "than the tracker's ~15 s bridge returns with a new track id, and "
                    "counting only one fragment "
                    "discarded the rest of their activity - measured once as cooking_duration_s "
                    "reading 0.00 beside a cooking segment of 8.6 s. Two tracks seen in the SAME "
                    "frame are never merged.",
                ] if subject_provenance == "asserted_disjoint_track_chain" else []),
            }
            t3 = time.time()
        except Exception as exc:                                   # noqa: BLE001
            # A demo that dies at stage 3 is worse than one that reports stage 3 failed.
            stage3 = {"error": f"{type(exc).__name__}: {exc}"}
            t3 = time.time()

    # AGENT 3 GOES OUT BEFORE QWEN IS CALLED. Generation is the slow stage - >125 s on a T4 -
    # so emitting Agent 3 first is the difference between a page that fills in progressively
    # and one that shows nothing for two minutes.
    yield "stage", _stage(3, "behaviour",
                          "done" if stage3 and "error" not in stage3
                          else ("failed" if stage3 else "skipped"),
                          t3 - t_cls, stage3 or {})

    t4 = t3
    if st is not None and stage3 and "error" not in stage3:
        if STATE.get("reporter") is None:
            stage4 = {"why_skipped": "no reporter is configured on this backend, so Agents 1-3 "
                                     "ran and the report was not written. Either the Qwen weights "
                                     "are unattached, or torch has no kernels for this GPU. Run "
                                     "Agents 3-4 yourself with /video?stages=12 - "
                                     "web/local_backend.py does that with local API keys."}
        else:
            try:
                out = None
                # HEARTBEAT THROUGH GENERATION - the longest silence in the whole request.
                # Agent 3's line used to go out and then nothing until Agent 4 finished, so a
                # normal Qwen pass tripped the client's 90 s silence watchdog and reported a
                # wedged GPU. Observed on a 5-second clip.
                for _kind, _obj in _blocking(lambda: STATE["reporter"].report(st),
                                             "qwen_generating"):
                    if _kind == "beat":
                        yield "heartbeat", _obj
                    else:
                        out = _obj
                rate = out.hallucination_rate
                stage4 = {
                    "model": out.report.model_name,
                    "constrained_decoding": out.report.constrained_decoding,
                    "summary": out.report.summary,
                    "recommendation": out.report.recommendation,
                    "escalate": out.report.escalate,
                    "claims_emitted": out.n_emitted_claims,
                    "claims_scorable": len(out.report.claims),
                    "hallucination_rate": None if rate != rate else round(rate, 3),
                    "parse_failed": out.parse_failed,
                    # WHY it did not parse. `_extract_json` distinguishes "no JSON object in
                    # response" (the model answered in prose - a compliance failure) from
                    # "unterminated JSON object (truncated generation)" (max_tokens too low for a
                    # model that reasons before answering - a budget failure). One boolean on the
                    # card cannot tell those apart, and they need different fixes.
                    "notes": list(out.notes),
                    "caveats": ["Claims cite features measured from this video; the baseline "
                                "they are compared against is the declared reference above."],
                }
                by_id = {c.claim_id: c for c in out.report.claims}
                for v in out.report.verifications:
                    c = by_id.get(v.claim_id)
                    checks.append({
                        "claim_id": v.claim_id,
                        "text": None if c is None else c.text,
                        "evidence_ref": None if c is None else c.evidence_ref,
                        "claimed_value": None if c is None else c.claimed_value,
                        "claimed_pct_change": None if c is None else c.claimed_pct_change,
                        "direction": None if c is None else c.direction,
                        # C1-C5 separately: "verified" as one boolean hides which guarantee
                        # is doing the work. C4 is the one that matters clinically; C5 is
                        # the one the latest measured run says catches the most - prose
                        # that never quotes the figure the field records.
                        "C1_ref_exists": v.ref_exists,
                        "C2_value_matches": v.value_matches,
                        "C3_pct_matches": v.pct_matches,
                        "C4_direction_consistent": v.direction_consistent,
                        "C5_prose_quoted_value": v.prose_quoted_value,
                        "faithful": v.is_faithful,
                        "notes": list(v.notes),
                        "shown_to_caregiver": v.is_faithful,
                    })
            except Exception as exc:                               # noqa: BLE001
                stage4 = {"error": f"{type(exc).__name__}: {exc}"}
    elif not tracks:
        stage4 = {"why_skipped": "no person was tracked long enough to fill a window, so "
                                 "there is nothing for the report to describe."}
    else:
        stage4 = {"why_skipped": "Agent 3 did not complete, so there is no verified state to "
                                 "write a report from."}
    t4 = time.time()
    _rss["after_report"] = _rss_mb()

    # Printed as well as returned, so it survives in the notebook log even if the client
    # disconnects mid-stream - which is exactly when a leak matters most.
    print("  RSS MB " + " -> ".join(f"{k} {v}" for k, v in _rss.items()))

    stage4_status = ("done" if stage4 and "error" not in stage4 and "why_skipped" not in stage4
                     else ("failed" if stage4 and "error" in stage4 else "skipped"))
    yield "stage", _stage(4, "report", stage4_status, t4 - t3, stage4 or {})

    stages = [stage1, stage2,
              _stage(3, "behaviour",
                     "done" if stage3 and "error" not in stage3
                     else ("failed" if stage3 else "skipped"), t3 - t_cls, stage3 or {}),
              _stage(4, "report", stage4_status, t4 - t3, stage4 or {})]

    yield "result", {
        "fps": poses["fps"], "width": poses["width"], "height": poses["height"],
        "frames_kept": poses["frames_kept"], "truncated": poses["truncated"],
        "reid": poses["reid"], "providers": poses["providers"],
        "frames": poses["frames"],          # per-frame skeletons, for the overlay
        "tracks": tracks,
        # People TRACKED, matching Agent 1's own card. `len(tracks)` would be people
        # CLASSIFIED, which is smaller whenever someone is present but too occluded to
        # label - and the overlay draws that person, so the count must not omit them.
        "n_people": len(tracked),
        "stages": stages,                   # Agent 1 -> 2 -> 3 -> 4, in order, with timings
        "checks": checks,                   # C1-C5 per claim, individually
        "timing": {"pose_s": round(t_pose - t_start, 1),
                   "classify_s": round(t_cls - t_pose, 1),
                   "behaviour_s": round(t3 - t_cls, 1),
                   "report_s": round(t4 - t3, 1)},
        # RSS AT EACH SEAM. The delta between consecutive entries is what a stage RETAINED, which
        # is the difference between a leak and a transient peak.
        "rss_mb": _rss,
        "tau": TAU, "temperature": TEMPERATURE,
        "report": (stage4 or {}).get("summary"),
        "report_provenance": (
            None if not (stage4 or {}).get("summary") else
            "Claims are verified against features measured from THIS video. The baseline "
            "they are compared to is a declared simulated reference, because a 14-day "
            "rolling median cannot come from one clip - see stages[2].caveats."),
    }

In [ ]:
# Serve, then tunnel. KEEP THIS CELL RUNNING - the address dies with the process.
import nest_asyncio, re, subprocess, threading, time, uvicorn
nest_asyncio.apply()

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True).start()
time.sleep(3)

tun = subprocess.Popen([str(CF), "tunnel", "--url", "http://localhost:8000",
                        "--no-autoupdate"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                       bufsize=1)
URL = None
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0)
        break
assert URL, "cloudflared exited without printing a URL - check the output above"

print()
print("=" * 68)
print("  PASTE THIS INTO THE FRONT END HEADER:")
print(f"    {URL}")
print("=" * 68)
print()
print("  /health   readiness + which streams loaded")
print("  /activity [N,30,2,17,3] windows -> smoothed labels")
print("  /report   a BehaviourState -> claims + C1-C4 verdicts")
print("  /video    an uploaded clip -> per-frame skeletons, tracks, roles, segments")
print()
print("Leave this cell running. Weights finish loading in the background - /health")
print("reports ready=false until then, and the front end says so rather than failing.")

# Block, so Kaggle does not treat the session as idle and reap it mid-demo.
tun.wait()